<a href="https://colab.research.google.com/github/niharikakt024/AI-Agent-for-Data-Cleaning/blob/main/DATA_CLEANING_AGENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install groq pandas -q

In [ ]:
!pip install thefuzz python-Levenshtein -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 15.0 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get('GROQ_API_KEY'))

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say hello in 5 words"}]
)
print(response.choices[0].message.content)

Hello, how are you today?


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print(f"Shape: {df.shape}")
df.head()

Shape: (100000, 8)


,Transaction_ID,Transaction_Date,Customer_ID,Product_Name,Quantity,Price,Payment_Method,Transaction_Status
0,T0001,2024-08-02,C2205,Headphones,-5.0,$420.21,pay pal,NaN
1,T0002,2020-02-10,C3156,Coffee,469.0,-445.34202525395585,creditcard,Pending
2,T0003,2025-02-30,C2919,Tablet,-4.0,810.9930123946459,credit card,completed
3,T0004,2020-08-17,C3009,Tab,-7.0,868.6083413217348,PayPal,Pending
4,T0005,2025-02-30,C3488,Coffee Machine,-10.0,-763.1224490039416,PayPal,completed


In [ ]:
def profile_dataframe(df):
    profile = {}
    for col in df.columns:
        profile[col] = {
            "dtype": str(df[col].dtype),
            "missing_count": int(df[col].isna().sum()),
            "missing_pct": round(df[col].isna().mean() * 100, 2),
            "unique_count": int(df[col].nunique()),
            "sample_values": df[col].dropna().astype(str).unique()[:8].tolist()
        }
    return profile

profile = profile_dataframe(df)
for col, info in profile.items():
    print(col, "->", info)
    print()

Transaction_ID -> {'dtype': 'object', 'missing_count': 5018, 'missing_pct': np.float64(5.02), 'unique_count': 94040, 'sample_values': ['T0001', 'T0002', 'T0003', 'T0004', 'T0005', 'T0006', 'T0008', 'T0009']}

Transaction_Date -> {'dtype': 'object', 'missing_count': 4880, 'missing_pct': np.float64(4.88), 'unique_count': 1861, 'sample_values': ['2024-08-02', '2020-02-10', '2025-02-30', '2020-08-17', '2021-10-26', '2023-13-01', '2020-03-18', '2020-06-19']}

Customer_ID -> {'dtype': 'object', 'missing_count': 4878, 'missing_pct': np.float64(4.88), 'unique_count': 5000, 'sample_values': ['C2205', 'C3156', 'C2919', 'C3009', 'C3488', 'C4241', 'C1313', 'C4736']}

Product_Name -> {'dtype': 'object', 'missing_count': 0, 'missing_pct': np.float64(0.0), 'unique_count': 46, 'sample_values': ['Headphones', 'Coffee ', 'Tablet', 'Tab', 'Coffee Machine', 'Smartphone', 'Laptop', 'Coffee Ma']}

Quantity -> {'dtype': 'float64', 'missing_count': 5019, 'missing_pct': np.float64(5.02), 'unique_count': 921, '

In [ ]:
import json

def get_cleaning_plan(profile):
    prompt = f"""You are a data cleaning agent. Given this column profile from a pandas DataFrame,
decide the best cleaning action for each column.

PROFILE:
{json.dumps(profile, indent=2, default=str)}

For each column, return a JSON object with this exact structure (no markdown, no explanation outside the JSON):

{{
  "column_name": {{
    "issue": "short description of the problem",
    "action": "one of: standardize_case | parse_dates | clean_currency | fill_missing_median | fill_missing_mode | drop_duplicates | none",
    "reasoning": "one sentence why"
  }}
}}

Return ONLY valid JSON, nothing else."""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )

    text = response.choices[0].message.content
    text = text.replace("```json", "").replace("```", "").strip()
    return json.loads(text)

plan = get_cleaning_plan(profile)
print(json.dumps(plan, indent=2))

{
  "Transaction_ID": {
    "issue": "missing values",
    "action": "fill_missing_mode",
    "reasoning": "This column has a relatively low missing value percentage and a high number of unique values, making mode imputation a suitable choice."
  },
  "Transaction_Date": {
    "issue": "invalid dates and missing values",
    "action": "parse_dates",
    "reasoning": "This column contains dates in a standard format, but some of them may be invalid, such as February 30, so parsing dates will help to identify and potentially correct these issues."
  },
  "Customer_ID": {
    "issue": "missing values",
    "action": "fill_missing_mode",
    "reasoning": "Similar to Transaction_ID, this column has a low missing value percentage and a high number of unique values, making mode imputation suitable."
  },
  "Product_Name": {
    "issue": "inconsistent naming conventions",
    "action": "standardize_case",
    "reasoning": "Product names have different cases and potentially duplicate names with 

In [ ]:
import pandas as pd
import numpy as np

def apply_cleaning_plan(df, plan):
    df_clean = df.copy()
    log = []

    for col, spec in plan.items():
        action = spec.get("action", "").lower()
        if col not in df_clean.columns:
            continue

        if "standardize_case" in action:
            df_clean[col] = df_clean[col].astype(str).str.strip().str.title()
            df_clean[col] = df_clean[col].replace("Nan", np.nan)
            log.append(col + ": standardized text casing")

        if "parse_dates" in action:
            df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce", format="mixed")
            log.append(col + ": parsed into consistent datetime format")

        if "clean_currency" in action:
            cleaned = df_clean[col].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False)
            df_clean[col] = pd.to_numeric(cleaned, errors="coerce")
            log.append(col + ": cleaned currency symbols, converted to numeric")

        if "fill_missing_median" in action:
            median_val = df_clean[col].median()
            df_clean[col] = df_clean[col].fillna(median_val)
            log.append(col + ": filled missing values with median (" + str(round(median_val, 2)) + ")")

        if "fill_missing_mode" in action:
            mode_val = df_clean[col].mode()[0]
            df_clean[col] = df_clean[col].fillna(mode_val)
            log.append(col + ": filled missing values with mode (" + str(mode_val) + ")")

    return df_clean, log

df_clean, log = apply_cleaning_plan(df, plan)

print("=== CLEANING LOG ===")
for entry in log:
    print("-", entry)

print("\n=== BEFORE (missing values) ===")
print(df.isna().sum())

print("\n=== AFTER (missing values) ===")
print(df_clean.isna().sum())

df_clean.head(10)

=== CLEANING LOG ===
- Transaction_ID: filled missing values with mode (T1258)
- Transaction_Date: parsed into consistent datetime format
- Customer_ID: filled missing values with mode (C2023)
- Product_Name: standardized text casing
- Quantity: filled missing values with median (6.0)
- Price: cleaned currency symbols, converted to numeric
- Payment_Method: standardized text casing
- Transaction_Status: standardized text casing

=== BEFORE (missing values) ===
Transaction_ID         5018
Transaction_Date       4880
Customer_ID            4878
Product_Name              0
Quantity               5019
Price                 33497
Payment_Method            0
Transaction_Status    16679
dtype: int64

=== AFTER (missing values) ===
Transaction_ID            0
Transaction_Date      68261
Customer_ID               0
Product_Name              0
Quantity                  0
Price                 33497
Payment_Method            0
Transaction_Status    16679
dtype: int64


,Transaction_ID,Transaction_Date,Customer_ID,Product_Name,Quantity,Price,Payment_Method,Transaction_Status
0,T0001,2024-08-02,C2205,Headphones,-5.0,420.210000,Pay Pal,NaN
1,T0002,2020-02-10,C3156,Coffee,469.0,-445.342025,Creditcard,Pending
2,T0003,NaT,C2919,Tablet,-4.0,810.993012,Credit Card,Completed
3,T0004,2020-08-17,C3009,Tab,-7.0,868.608341,Paypal,Pending
4,T0005,NaT,C3488,Coffee Machine,-10.0,-763.122449,Paypal,Completed
5,T0006,2021-10-26,C4241,Smartphone,598.0,NaN,Paypal,Completed
6,T1258,NaT,C1313,Laptop,10.0,NaN,Credit Card,Completed
7,T0008,NaT,C4736,Headphones,669.0,-86.921269,Cash,NaN
8,T0009,NaT,C3387,Tablet,10.0,461.701984,Paypal,NaN
9,T0010,NaT,C2846,Laptop,-1.0,404.890707,Creditcard,Pending


In [ ]:
from thefuzz import fuzz

def auto_standardize_categories(df, column, similarity_threshold=80):
    """
    Automatically groups near-duplicate category values in a column
    (e.g. 'Tab' vs 'Tablet', 'Pay Pal' vs 'Paypal') and maps them
    to the most frequent spelling in each group.
    Works on any messy categorical column.
    """
    series = df[column].astype(str).str.strip()
    value_counts = series.value_counts()
    unique_vals = value_counts.index.tolist()

    mapping = {}
    assigned = set()

    for val in unique_vals:
        if val in assigned:
            continue
        # this value becomes the "canonical" name for its group
        group = [val]
        assigned.add(val)

        for other in unique_vals:
            if other in assigned:
                continue
            similarity = fuzz.token_sort_ratio(val.lower(), other.lower())
            if similarity >= similarity_threshold:
                group.append(other)
                assigned.add(other)

        # pick the most common spelling in the group as the standard
        canonical = value_counts[group].idxmax()
        for member in group:
            mapping[member] = canonical

    df[column] = series.map(mapping)
    return df, mapping


# Apply to any messy text column — works generically
for col in ["Payment_Method", "Product_Name"]:
    df_clean, mapping_used = auto_standardize_categories(df_clean, col)
    print(f"\n{col} — groups detected:")
    for original, standardized in mapping_used.items():
        if original != standardized:
            print(f"  '{original}' -> '{standardized}'")

print("\n=== FINAL VALUE COUNTS ===")
print(df_clean["Payment_Method"].value_counts())
print(df_clean["Product_Name"].value_counts())


Payment_Method — groups detected:
  'credit card' -> 'Credit Card'

Product_Name — groups detected:
  'Table' -> 'Tablet'
  'Tabl' -> 'Tablet'
  'Lapt' -> 'Laptop'
  'Lapto' -> 'Laptop'
  'Smartph' -> 'Smartphone'
  'Smartphon' -> 'Smartphone'
  'Smartpho' -> 'Smartphone'
  'Coffee Machi' -> 'Coffee Machine'
  'Coffee Mach' -> 'Coffee Machine'
  'Coffee Machin' -> 'Coffee Machine'
  'Coffee Mac' -> 'Coffee Machine'
  'Headpho' -> 'Headphones'
  'Headphon' -> 'Headphones'
  'Headphone' -> 'Headphones'
  'Lap' -> 'La'
  'Tab' -> 'Ta'
  'Coffee Ma' -> 'Coffee'
  'Coffee M' -> 'Coffee'
  'Coffe' -> 'Coffee'
  'Coff' -> 'Coffee'
  'Hea' -> 'He'
  'Smartp' -> 'Smart'
  'Smar' -> 'Smart'
  'Head' -> 'Headp'
  'Headph' -> 'Headp'
  'Sm' -> 'Sma'
  'Co' -> 'Cof'

=== FINAL VALUE COUNTS ===
Payment_Method
Credit Card    26959
PayPal         26705
pay pal        13535
creditcard     13494
Cash           13348
Name: count, dtype: int64
Product_Name
Tablet            18038
Laptop            17828


/tmp/ipykernel_581/1039621773.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column] = series.map(mapping)
/tmp/ipykernel_581/1039621773.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column] = series.map(mapping)


In [ ]:
dupes = df_clean[df_clean.duplicated(subset=["EmployeeID"], keep=False)]
print(f"Duplicate rows found: {len(dupes)}")
dupes.sort_values("EmployeeID").head(10)

Duplicate rows found: 40


,EmployeeID,Name,Department,Gender,JoinDate,Age,Salary
27,28,Employee_28,HR,Female,2021-08-22,32.9,68499.73
507,28,Employee_28,HR,Female,2021-08-22,32.9,68499.73
502,38,Employee_38,HR,Female,2023-01-15,42.7,76322.55
37,38,Employee_38,HR,Female,2023-01-15,42.7,76322.55
87,88,Employee_88,Sales,Male,2018-05-16,44.1,92427.08
517,88,Employee_88,Sales,Male,2018-05-16,44.1,92427.08
518,113,Employee_113,Engineering,Male,2023-03-18,28.3,43293.99
112,113,Employee_113,Engineering,Male,2023-03-18,28.3,43293.99
130,131,Employee_131,HR,Female,2024-10-13,23.1,83353.07
500,131,Employee_131,HR,Female,2024-10-13,23.1,83353.07


In [ ]:
df_clean = df.copy()

In [ ]:
before_count = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["Transaction_ID"], keep="first")
after_count = len(df_clean)

print(f"Removed {before_count - after_count} duplicate rows")
print(f"Final shape: {df_clean.shape}")

Removed 5959 duplicate rows
Final shape: (94041, 8)


In [ ]:
def generate_report(df_before, df_after, log, dupes_removed):
    report_lines = []
    report_lines.append("=" * 50)
    report_lines.append("DATA CLEANING AGENT — SUMMARY REPORT")
    report_lines.append("=" * 50)
    report_lines.append(f"\nOriginal rows: {len(df_before)}")
    report_lines.append(f"Final rows: {len(df_after)}")
    report_lines.append(f"Duplicate rows removed: {dupes_removed}")
    report_lines.append(f"\nTotal missing values before: {df_before.isna().sum().sum()}")
    report_lines.append(f"Total missing values after: {df_after.isna().sum().sum()}")
    report_lines.append("\nActions taken:")
    for entry in log:
        report_lines.append(f"  - {entry}")
    report_lines.append(f"  - Gender standardized to Male/Female")
    report_lines.append(f"  - Department standardized to Sales/Marketing/Engineering/HR")
    report_lines.append(f"  - {dupes_removed} exact duplicate rows removed")
    report_lines.append("\n" + "=" * 50)
    return "\n".join(report_lines)

report = generate_report(df, df_clean, log, before_count - after_count)
print(report)

# Save the cleaned data + report
df_clean.to_csv("cleaned_data.csv", index=False)
with open("cleaning_report.txt", "w") as f:
    f.write(report)

from google.colab import files
files.download("cleaned_data.csv")
files.download("cleaning_report.txt")

DATA CLEANING AGENT — SUMMARY REPORT

Original rows: 100000
Final rows: 94041
Duplicate rows removed: 5959

Total missing values before: 69971
Total missing values after: 60963

Actions taken:
  - Transaction_ID: filled missing values with mode (T1258)
  - Transaction_Date: parsed into consistent datetime format
  - Customer_ID: filled missing values with mode (C2023)
  - Product_Name: standardized text casing
  - Quantity: filled missing values with median (6.0)
  - Price: cleaned currency symbols, converted to numeric
  - Payment_Method: standardized text casing
  - Transaction_Status: standardized text casing
  - Gender standardized to Male/Female
  - Department standardized to Sales/Marketing/Engineering/HR
  - 5959 exact duplicate rows removed



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>